In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False,  json.loads(sys.argv[1])['scenario'])

# START

In [2]:
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import connectionscanmodel, stepmodel
from quetzal.io import excel
import numpy as np
import pandas as pd
import geopandas as gpd

import shapely

import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

In [5]:
PARKING_FARES = var['fare_parking'].to_dict()

In [6]:
mpath + 'all_networks'

'../../scenarios/base/models/all_networks'

# READ

In [7]:
sm = stepmodel.read_zippedpickles(mpath + 'all_networks')

walk_on_road: 100%|██████████| 11/11 [00:00<00:00, 27.24it/s]


### Read zones, fix parking fares

In [ ]:
zones = gpd.read_file(ipath + 'zones/final/zonage.gpkg').to_crs(epsg='2154')[['NUM_ZONE', 'NOM', 'parking_zone', 'lat_centroid', 'lng_centroid', 'nb household', 'nb workplaces', 'geometry']]
# zones = gpd.read_file(ipath + r"MT_/Zones_Gironde_BX.gpkg").to_crs(epsg='2154')
zones['index'] = 'zone_' + zones['NUM_ZONE'].apply(lambda x: f'0{x}' if x<10 else str(x))
zones['parking_fare'] = zones['parking_zone'].map(PARKING_FARES)

In [9]:
zones.set_index('index', inplace=True)
sm.zones = zones.copy()

centroids = zones.copy()
centroids['geometry'] = centroids['geometry'].apply(lambda x: x.centroid)
sm.centroids = centroids.copy()

In [10]:
### ### Tweak Libourne location
### OD_test_points = gpd.read_file('../../inputs/zones/calage_OD.gpkg').to_crs(epsg=2154)
### sm.centroids['geometry'] = sm.centroids.apply(lambda x: OD_test_points.set_index('NUM_ZONE')['geometry'].to_dict().get(x['NUM_ZONE'], x['geometry']), 1)

# EXPORT

In [11]:
sm.to_zippedpickles(
    mpath + r'zones'
)

centroids: 100%|██████████| 13/13 [00:00<00:00, 13.96it/s] 


In [12]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined